In [ ]:
"""
gridMET fire weather climatology by Pyrome X FOD fire location
Exports daily variables across a 15-year time-period within a buffer of fire points

author: maxwell.cook@colostate.edu
"""

import ee
import geemap
import time

ee.Authenticate()
ee.Initialize(project='cfri-ee')
print("GEE Authenticated !")

In [ ]:
# --- Load state boundaries, grab CO
states = ee.FeatureCollection("TIGER/2018/States")
co = states.filter(ee.Filter.eq('NAME','Colorado'))
# --- Load Pyromes FeatureCollection
pyromes = ee.FeatureCollection('projects/cfri-ee/assets/weather/Pyromes_CONUS_20200206')
# --- Filter to Colorado Pyromes
pyromes = pyromes.filter(ee.Filter.bounds(co))
pyromes.limit(10)

In [ ]:
# --- Load the GridMET ImageCollection
gridmet = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
print(f"gridMET bands:\n\n{gridmet.first().bandNames().getInfo()}")

In [ ]:
# --- Load the fire data (for sampling weather climatology)
# FPA-FOD, Size Classes D/E/F/G (>= 100 acres) in CO pyromes
fod = ee.FeatureCollection('projects/cfri-ee/assets/weather/Fires_ClassDEFG_CO_Pyromes')
print(f"Number of fire points (D,E,F,G): {fod.size().getInfo()}")
print(
    fod.aggregate_min('FIRE_YEAR').getInfo(),
    fod.aggregate_max('FIRE_YEAR').getInfo()
)

# Spatial join: stamp each FOD point with the pyrome it falls in
def add_PYROME(pt):
    pyrome_match = pyromes.filterBounds(pt.geometry()).first()
    return pt.set('PYROME', pyrome_match.get('PYROME'))

fod_pyrome = fod.map(add_PYROME)

# Build a fire-buffered mask per pyrome (5 km circle around each FOD point).
# Dissolving per pyrome gives one polygon per pyrome whose footprint captures
# the fire-prone terrain — mimicking the empirical siting bias of RAWS toward
# fire-active mid-slope environments. The subsequent reducer is evaluated over
# this mask rather than the full pyrome, so irrigated ag, alpine tundra, and
# other non-fire-relevant cells do not dilute the climatology.
BUFFER_M = 5000  # 5 km; ecologically captures adjacent fire-prone terrain

def buffer_fire(ftr):
    return ftr.buffer(BUFFER_M)

fod_buffered = fod_pyrome.map(buffer_fire)

# Dissolve FOD buffers per pyrome, then intersect with pyrome boundary so a
# buffer that straddles a pyrome edge does not leak into neighbours.
def fire_mask_for_pyrome(pyrome):
    pid = pyrome.get('PYROME')
    fires_in = fod_buffered.filter(ee.Filter.eq('PYROME', pid))
    fire_union = fires_in.geometry(maxError=1)
    clipped = pyrome.geometry().intersection(fire_union, 1)
    return pyrome.setGeometry(clipped)

pyrome_fire_mask = pyromes.map(fire_mask_for_pyrome)

print(f"Built fire mask for {pyrome_fire_mask.size().getInfo()} pyromes")
print("  Each feature's geometry = union of 5-km fire buffers ∩ pyrome boundary.")

In [ ]:
"""
Filter GridMET to fire season 2010-2025 and reduce over the fire-mask geometry
using a multi-percentile reducer.

Per-day reduction: for each daily image, compute the [10, 25, 50, 75, 90]
percentile of each variable across all GridMET cells intersecting the
fire mask.  Output columns carry a `_pXX` suffix (e.g., `erc_p75`).
Downstream, `fb_tools.weather.gridmet.load_gridmet_csv(tail_percentile=...)`
selects the fire-dangerous tail at analysis time — e.g., p75 of hot/dry
variables (tmmx, erc, vs, vpd) paired with p25 of moisture variables
(fm100, rmin, rmax) approximates the driest quartile of cells within the
fire mask.
"""

# --- Select bands
gridmet = gridmet.select(['vpd','erc','fm100','fm1000','tmmx','tmmn','rmax','rmin','pr','vs','th'])

gridmet_fs = gridmet.filter(ee.Filter.And(
    ee.Filter.calendarRange(2010, 2025, 'year'),
    ee.Filter.calendarRange(4, 10, 'month')
)).filterBounds(pyromes)

n_images = gridmet_fs.size().getInfo()
n_pyromes = pyrome_fire_mask.size().getInfo()
print(f"GridMET fire-season images: {n_images}  (expect ~16yrs × 214days = 3424)")
print(f"Pyromes in analysis area:   {n_pyromes}")
print(f"Expected output features:   ~{n_images * n_pyromes:,}")

# Multi-percentile reducer: returns `<band>_p10`, `<band>_p25`, `<band>_p50`,
# `<band>_p75`, `<band>_p90` per variable per feature.
percentile_reducer = ee.Reducer.percentile([10, 25, 50, 75, 90])

image_list = gridmet_fs.toList(gridmet_fs.size())

def extract_day(img):
    img = ee.Image(img)
    date_str = img.date().format('YYYY-MM-dd')
    year = img.date().get('year')
    doy = img.date().getRelative('day', 'year').add(1)  # 1-indexed DOY

    sampled = img.reduceRegions(
        collection=pyrome_fire_mask,
        reducer=percentile_reducer,
        scale=4000
    ).map(lambda f: f.set({
        'PYROME': f.get('PYROME'),
        'date': date_str,
        'year': year,
        'doy': doy,
    }))
    return sampled

all_samples_fc = ee.FeatureCollection(image_list.map(extract_day)).flatten()

# Retain meta columns + all percentile-suffix variable columns.
BANDS = ['vpd','erc','fm100','fm1000','tmmx','tmmn','rmax','rmin','pr','vs','th']
PCTLS = [10, 25, 50, 75, 90]
keep_cols = ['PYROME', 'date', 'year', 'doy'] + [f'{b}_p{p}' for b in BANDS for p in PCTLS]
all_samples_fc = all_samples_fc.select(keep_cols)
print(f"FeatureCollection ready. {len(keep_cols)} columns "
      f"({len(BANDS)} bands × {len(PCTLS)} percentiles + 4 meta).")
print("  NOTE: tmmx/tmmn in °K — fb_tools.weather.gridmet.load_gridmet_csv() converts to °F.")

In [ ]:
# --- Export to Google Drive

def drop_geometry(feature):
    return feature.setGeometry(None)

all_samples_fc = all_samples_fc.map(drop_geometry)

# New filename distinguishes this percentile-suffix export from the legacy
# pyrome-median CSV (`gridmet_clim_CO_pyromes.csv`).
export_task = ee.batch.Export.table.toDrive(
    collection=all_samples_fc,
    description='gridmet_clim_CO_pyromes_fmask_pctiles',
    folder='fb_tools_weather',
    fileNamePrefix='gridmet_clim_CO_pyromes_fmask_pctiles',
    fileFormat='CSV'
)

export_task.start()
print("Export to Google Drive started: gridmet_clim_CO_pyromes_fmask_pctiles.csv")
print("  Load with: load_gridmet_csv(path, tail_percentile=75)")